# OpenCV Document Scanner Notebook

이 노트북은 저장소에 포함된 `scan.py` 스크립트의 기능을 주피터 노트북 환경에서 사용할 수 있도록 재구성한 것입니다. 아래 셀을 순서대로 실행하면 이미지 또는 이미지 폴더에 대해 문서 스캔(투시 보정 + 이진화)을 수행할 수 있습니다.

> **참고:** 노트북이 실행되는 환경에 OpenCV, SciPy, matplotlib, pylsd 등의 종속성이 설치되어 있어야 합니다. 필요하다면 `requirements.txt` 파일을 참고해 설치하세요.

In [ ]:
# 필수 라이브러리를 불러옵니다.
from pathlib import Path
from itertools import combinations

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from pylsd.lsd import lsd
from scipy.spatial import distance as dist

from pyimagesearch import imutils, transform
import polygon_interacter as poly_i

## DocScanner 클래스 정의

원본 `scan.py` 파일에서 사용하던 `DocScanner` 클래스를 거의 동일하게 옮겨 왔습니다. 인터랙티브 모드(문서 꼭짓점을 수동으로 조정) 역시 matplotlib 위젯을 통해 지원합니다.

In [ ]:
class DocScanner:
    """OpenCV를 활용한 간단한 문서 스캐너"""

    def __init__(self, interactive: bool = False, *, MIN_QUAD_AREA_RATIO: float = 0.25, MAX_QUAD_ANGLE_RANGE: int = 40):
        self.interactive = interactive
        self.MIN_QUAD_AREA_RATIO = MIN_QUAD_AREA_RATIO
        self.MAX_QUAD_ANGLE_RANGE = MAX_QUAD_ANGLE_RANGE

    # ----- Geometry helpers -----
    def filter_corners(self, corners, min_dist: float = 20):
        def predicate(representatives, corner):
            return all(dist.euclidean(representative, corner) >= min_dist for representative in representatives)

        filtered_corners = []
        for c in corners:
            if predicate(filtered_corners, c):
                filtered_corners.append(c)
        return filtered_corners

    def angle_between_vectors_degrees(self, u, v):
        return np.degrees(np.arccos(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))))

    def get_angle(self, p1, p2, p3):
        a = np.radians(np.array(p1))
        b = np.radians(np.array(p2))
        c = np.radians(np.array(p3))

        avec = a - b
        cvec = c - b
        return self.angle_between_vectors_degrees(avec, cvec)

    def angle_range(self, quad):
        tl, tr, br, bl = quad
        ura = self.get_angle(tl[0], tr[0], br[0])
        ula = self.get_angle(bl[0], tl[0], tr[0])
        lra = self.get_angle(tr[0], br[0], bl[0])
        lla = self.get_angle(br[0], bl[0], tl[0])
        return np.ptp([ura, ula, lra, lla])

    # ----- Corner / contour detection -----
    def get_corners(self, img):
        corners = []
        lines = lsd(img)

        if lines is not None:
            lines = lines.squeeze().astype(np.int32).tolist()
            horizontal_lines_canvas = np.zeros(img.shape, dtype=np.uint8)
            vertical_lines_canvas = np.zeros(img.shape, dtype=np.uint8)

            for line in lines:
                x1, y1, x2, y2, _ = line
                if abs(x2 - x1) > abs(y2 - y1):
                    (x1, y1), (x2, y2) = sorted(((x1, y1), (x2, y2)), key=lambda pt: pt[0])
                    cv2.line(horizontal_lines_canvas, (max(x1 - 5, 0), y1), (min(x2 + 5, img.shape[1] - 1), y2), 255, 2)
                else:
                    (x1, y1), (x2, y2) = sorted(((x1, y1), (x2, y2)), key=lambda pt: pt[1])
                    cv2.line(vertical_lines_canvas, (x1, max(y1 - 5, 0)), (x2, min(y2 + 5, img.shape[0] - 1)), 255, 2)

            lines = []

            contours, _ = cv2.findContours(horizontal_lines_canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            contours = sorted(contours, key=lambda c: cv2.arcLength(c, True), reverse=True)[:2]
            horizontal_lines_canvas = np.zeros(img.shape, dtype=np.uint8)
            for contour in contours:
                contour = contour.reshape((contour.shape[0], contour.shape[2]))
                min_x = np.amin(contour[:, 0], axis=0) + 2
                max_x = np.amax(contour[:, 0], axis=0) - 2
                left_y = int(np.average(contour[contour[:, 0] == min_x][:, 1]))
                right_y = int(np.average(contour[contour[:, 0] == max_x][:, 1]))
                lines.append((min_x, left_y, max_x, right_y))
                cv2.line(horizontal_lines_canvas, (min_x, left_y), (max_x, right_y), 1, 1)
                corners.append((min_x, left_y))
                corners.append((max_x, right_y))

            contours, _ = cv2.findContours(vertical_lines_canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            contours = sorted(contours, key=lambda c: cv2.arcLength(c, True), reverse=True)[:2]
            vertical_lines_canvas = np.zeros(img.shape, dtype=np.uint8)
            for contour in contours:
                contour = contour.reshape((contour.shape[0], contour.shape[2]))
                min_y = np.amin(contour[:, 1], axis=0) + 2
                max_y = np.amax(contour[:, 1], axis=0) - 2
                top_x = int(np.average(contour[contour[:, 1] == min_y][:, 0]))
                bottom_x = int(np.average(contour[contour[:, 1] == max_y][:, 0]))
                lines.append((top_x, min_y, bottom_x, max_y))
                cv2.line(vertical_lines_canvas, (top_x, min_y), (bottom_x, max_y), 1, 1)
                corners.append((top_x, min_y))
                corners.append((bottom_x, max_y))

            corners_y, corners_x = np.where(horizontal_lines_canvas + vertical_lines_canvas == 2)
            corners += list(zip(corners_x, corners_y))

        return self.filter_corners(corners)

    def is_valid_contour(self, cnt, im_width, im_height):
        return (
            len(cnt) == 4
            and cv2.contourArea(cnt) > im_width * im_height * self.MIN_QUAD_AREA_RATIO
            and self.angle_range(cnt) < self.MAX_QUAD_ANGLE_RANGE
        )

    def get_contour(self, rescaled_image):
        MORPH = 9
        CANNY = 84
        IM_HEIGHT, IM_WIDTH, _ = rescaled_image.shape

        gray = cv2.cvtColor(rescaled_image, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (7, 7), 0)

        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (MORPH, MORPH))
        dilated = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)
        edged = cv2.Canny(dilated, 0, CANNY)

        test_corners = self.get_corners(edged)
        approx_contours = []

        if len(test_corners) >= 4:
            quads = []
            for quad in combinations(test_corners, 4):
                points = np.array(quad)
                points = transform.order_points(points)
                points = np.array([[p] for p in points], dtype="int32")
                quads.append(points)

            quads = sorted(quads, key=cv2.contourArea, reverse=True)[:5]
            quads = sorted(quads, key=self.angle_range)

            approx = quads[0]
            if self.is_valid_contour(approx, IM_WIDTH, IM_HEIGHT):
                approx_contours.append(approx)

        cnts, _ = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:5]

        for c in cnts:
            approx = cv2.approxPolyDP(c, 80, True)
            if self.is_valid_contour(approx, IM_WIDTH, IM_HEIGHT):
                approx_contours.append(approx)
                break

        if not approx_contours:
            screen_cnt = np.array(
                [
                    [IM_WIDTH, 0],
                    [IM_WIDTH, IM_HEIGHT],
                    [0, IM_HEIGHT],
                    [0, 0],
                ]
            )
        else:
            screen_cnt = max(approx_contours, key=cv2.contourArea).reshape(4, 2)

        return screen_cnt

    def interactive_get_contour(self, screen_cnt, rescaled_image):
        poly = Polygon(screen_cnt, animated=True, fill=False, color="yellow", linewidth=5)
        fig, ax = plt.subplots()
        ax.add_patch(poly)
        ax.set_title("Drag the corners of the box to the corners of the document.
Close the window when finished.")
        p = poly_i.PolygonInteractor(ax, poly)
        plt.imshow(rescaled_image)
        plt.show()
        new_points = p.get_poly_points()[:4]
        new_points = np.array([[p] for p in new_points], dtype="int32")
        return new_points.reshape(4, 2)

    def scan(self, image_path, *, output_dir: Path = Path("output")):
        RESCALED_HEIGHT = 500.0
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        image = cv2.imread(str(image_path))
        if image is None:
            raise ValueError(f"이미지를 불러올 수 없습니다: {image_path}")

        ratio = image.shape[0] / RESCALED_HEIGHT
        orig = image.copy()
        rescaled_image = imutils.resize(image, height=int(RESCALED_HEIGHT))

        screen_cnt = self.get_contour(rescaled_image)
        if self.interactive:
            screen_cnt = self.interactive_get_contour(screen_cnt, rescaled_image)

        warped = transform.four_point_transform(orig, screen_cnt * ratio)
        gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)

        sharpen = cv2.GaussianBlur(gray, (0, 0), 3)
        sharpen = cv2.addWeighted(gray, 1.5, sharpen, -0.5, 0)

        thresh = cv2.adaptiveThreshold(
            sharpen, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 15
        )

        basename = Path(image_path).name
        output_path = output_dir / basename
        cv2.imwrite(str(output_path), thresh)
        print(f"Processed {basename} -> {output_path}")
        return output_path

## 헬퍼 함수: 단일 이미지 또는 폴더 처리

원하는 이미지 경로를 지정하거나 폴더 전체를 한 번에 처리할 수 있도록 도우미 함수를 제공합니다.

In [ ]:
def scan_single_image(image_path, *, interactive=False, output_dir="output"):
    scanner = DocScanner(interactive=interactive)
    return scanner.scan(Path(image_path), output_dir=Path(output_dir))


def scan_image_folder(folder_path, *, interactive=False, output_dir="output", valid_exts=None):
    if valid_exts is None:
        valid_exts = {".jpg", ".jpeg", ".jp2", ".png", ".bmp", ".tiff", ".tif"}

    scanner = DocScanner(interactive=interactive)
    folder_path = Path(folder_path)
    results = []

    for file_path in sorted(folder_path.iterdir()):
        if file_path.suffix.lower() in valid_exts:
            results.append(scanner.scan(file_path, output_dir=Path(output_dir)))

    return results

## 사용 예시

아래 셀의 주석을 해제하고 경로를 원하는 값으로 바꿔 실행하면 됩니다. 예시는 저장소에 포함된 `sample_images` 폴더를 대상으로 합니다.

In [ ]:
# 예시: 단일 이미지 스캔
# scan_single_image('sample_images/desk.JPG', interactive=False)

# 예시: 폴더 내 모든 이미지 스캔
# scan_image_folder('sample_images', interactive=False)